# Análise de Acidentes de Trânsito no Brasil

Projeto G2 - Tema 27. Este notebook apresenta uma análise exploratória da base simulada de acidentes de trânsito no Brasil entre 2015 e 2024.

## 1. Contextualização

Acidentes de trânsito geram mortes, ferimentos, congestionamentos e custos sociais. A análise de dados permite identificar padr?es e apoiar campanhas educativas e políticas públicas.

## 2. Explicação da base

A base contém ano, mês, data, região, estado, cidade, tipo de acidente, período do dia, acidentes, feridos, mortes, chuva, visibilidade, veículos envolvidos e nível de gravidade.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dados/simulacao_acidentes_transito_brasil.csv", parse_dates=["data"])
df.head()

## 3. Limpeza e preparação

A base simulada já foi gerada em formato tabular limpo. Mesmo assim, são conferidos tipos, valores ausentes e criado um indicador de taxa de gravidade.

In [ ]:
df.info()
df.isna().sum()

In [ ]:
df["taxa_gravidade"] = (df["mortes"] * 5 + df["feridos"] * 1.5) / df["acidentes"].clip(lower=1)
df["ano_mes"] = df["data"].dt.to_period("M").astype(str)
df[["acidentes", "feridos", "mortes", "taxa_gravidade"]].describe()

## 4. KPIs


In [ ]:
kpis = {
    "Total de acidentes": int(df["acidentes"].sum()),
    "Total de mortes": int(df["mortes"].sum()),
    "Estado mais cr?tico": df.groupby("uf")["acidentes"].sum().idxmax(),
    "Tipo predominante": df.groupby("tipo_acidente")["acidentes"].sum().idxmax(),
    "Hor?rio mais perigoso": df.groupby("periodo_dia")["mortes"].sum().idxmax(),
    "Taxa m?dia de gravidade": round(df["taxa_gravidade"].mean(), 2),
}
kpis

## 5. Visualizações


In [ ]:
temporal = df.groupby("ano", as_index=False)["acidentes"].sum()
plt.figure(figsize=(10,4))
sns.lineplot(data=temporal, x="ano", y="acidentes", marker="o")
plt.title("Evolução anual dos acidentes")
plt.show()

In [ ]:
estado = df.groupby("uf", as_index=False)["acidentes"].sum().sort_values("acidentes", ascending=False)
plt.figure(figsize=(12,5))
sns.barplot(data=estado, x="uf", y="acidentes", color="#b42318")
plt.title("Acidentes por estado")
plt.show()

In [ ]:
tipo = df.groupby("tipo_acidente", as_index=False)["acidentes"].sum().sort_values("acidentes", ascending=False)
plt.figure(figsize=(8,4))
sns.barplot(data=tipo, x="tipo_acidente", y="acidentes", palette="Set2")
plt.title("Ranking por tipo de acidente")
plt.xticks(rotation=15)
plt.show()

In [ ]:
heat = df.pivot_table(index="periodo_dia", columns="nivel_gravidade", values="acidentes", aggfunc="sum", fill_value=0)
plt.figure(figsize=(8,4))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="OrRd")
plt.title("Acidentes por horário e gravidade")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="chuva_mm", y="acidentes", hue="visibilidade", alpha=0.7)
plt.title("Relação entre chuva e acidentes")
plt.show()

## 6. Interpretação

Os maiores volumes tendem a se concentrar em estados mais populosos e economicamente intensos. O período noturno e a madrugada merecem atenção porque combinam menor visibilidade, fadiga e maior severidade potencial. A chuva e a visibilidade ruim não explicam todos os acidentes, mas aparecem como fatores relevantes para priorização de prevenção.

## 7. Conclusão

O projeto mostra que decisões de segurança viária devem priorizar regiões críticas, horários de maior risco e fatores ambientais. O dashboard em Streamlit permite explorar esses padrões por filtros e comunicar resultados de forma clara.